# CUB-200: EfficientNet-B4

Trains `efficientnet_b4` on five stratified CUB-200 folds, evaluates each fold, and runs attribution agreement. Attach the updated `spinexnet-code` dataset and the CUB dataset.

In [ ]:
%pip install -q timm captum grad-cam scikit-learn scipy seaborn

# CUB-200 per-model notebook. Run one model per Kaggle session.
SELECTED_MODEL = "efficientnet_b4"

from pathlib import Path
import json, os, shutil, subprocess, sys
import pandas as pd

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
PY = sys.executable

FOLDS = [0, 1, 2, 3, 4]
XAI_FOLDS = [0, 1, 2, 3, 4]
MAX_SAMPLES = 300
PARALLEL_FOLDS_IF_2GPU = True
SAVE_MAPS = True
RUN_FAITHFULNESS = True

BATCH = {
    "convnext_blackbox": 32,
    "resnet50": 32,
    "densenet121": 32,
    "efficientnet_b4": 16,
    "vit_small": 16,
    "deit_small": 16,
}
EPOCHS = {
    "convnext_blackbox": 35,
    "resnet50": 15,
    "densenet121": 15,
    "efficientnet_b4": 15,
    "vit_small": 15,
    "deit_small": 30,
}
LR = {
    "convnext_blackbox": 1e-4,
    "resnet50": 3e-4,
    "densenet121": 3e-4,
    "efficientnet_b4": 3e-4,
    "vit_small": 3e-4,
    "deit_small": 1e-4,
}
BACKBONE_LR = {
    "convnext_blackbox": 5e-5,
    "deit_small": 5e-5,
}
HEAD_LR = {
    "convnext_blackbox": 5e-4,
    "deit_small": 5e-4,
}
WARMUP_EPOCHS = {
    "convnext_blackbox": 3,
    "deit_small": 3,
}
PATIENCE = {
    "convnext_blackbox": 8,
    "deit_small": 8,
}
DROP_PATH = {
    "convnext_blackbox": 0.1,
    "deit_small": 0.1,
}
METHODS = [
    "gradcam",
    "gradcam++",
    "integrated_gradients",
    "gradient_shap",
    "occlusion",
    "guided_backprop",
]

def run(cmd, env=None):
    print("\n$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True, env=env)

def input_roots():
    roots = [WORK]
    if INPUT.exists():
        roots += [p for p in INPUT.iterdir() if p.is_dir()]
        datasets = INPUT / "datasets"
        if datasets.exists():
            for owner in datasets.iterdir():
                if owner.is_dir():
                    roots += [p for p in owner.iterdir() if p.is_dir()]
    return roots

def looks_like_code(p):
    return (p / "cub_200_generalization" / "train_cub_model.py").exists()

def find_code_source():
    candidates = [INPUT / "spinexnet-code", INPUT / "spinexnet-code" / "het-spine"]
    for r in input_roots():
        candidates += [r, r / "het-spine", r / "spinexnet-code"]
    for p in candidates:
        if looks_like_code(p):
            return p
    raise FileNotFoundError("Could not find spinexnet-code with cub_200_generalization/train_cub_model.py")

def find_cub_root():
    candidates = []
    for r in input_roots():
        candidates += [r / "CUB_200_2011"]
        candidates += list(r.glob("**/CUB_200_2011"))
    for p in candidates:
        if (p / "images.txt").exists() and (p / "images").exists():
            return p
    raise FileNotFoundError("Could not find CUB_200_2011. Attach the CUB dataset to this notebook.")

def gpu_count():
    try:
        import torch
        return torch.cuda.device_count()
    except Exception:
        return 0

def run_jobs(jobs, parallel_if_2gpu=True):
    n_gpu = gpu_count()
    if parallel_if_2gpu and n_gpu >= 2 and len(jobs) > 1:
        for start in range(0, len(jobs), n_gpu):
            group = jobs[start:start+n_gpu]
            procs = []
            for local_idx, (name, cmd) in enumerate(group):
                env = os.environ.copy()
                env["CUDA_VISIBLE_DEVICES"] = str(local_idx)
                env["PYTHONUNBUFFERED"] = "1"
                print("\n$", " ".join(map(str, cmd)), f"  # {name} on visible GPU {local_idx}", flush=True)
                procs.append((name, subprocess.Popen(list(map(str, cmd)), env=env)))
            failures = []
            for name, proc in procs:
                rc = proc.wait()
                if rc != 0:
                    failures.append((name, rc))
            if failures:
                raise subprocess.CalledProcessError(failures[0][1], failures[0][0])
    else:
        for name, cmd in jobs:
            run(cmd)

def safe_same_path(a, b):
    try:
        return Path(a).resolve() == Path(b).resolve()
    except Exception:
        return False

def merge_previous_cub_outputs():
    for tree_name in ["cub_outputs", "cub_eval", "cub_xai"]:
        dest = WORK / tree_name
        for root in input_roots():
            src = root / tree_name
            if src.exists() and not safe_same_path(src, dest):
                print(f"Merging prior {tree_name}: {src} -> {dest}", flush=True)
                shutil.copytree(src, dest, dirs_exist_ok=True)

SRC = find_code_source()
CODE = WORK / "spinexnet-code"
if SRC.resolve() != CODE.resolve():
    shutil.copytree(SRC, CODE, dirs_exist_ok=True)
CUB_CODE = CODE / "cub_200_generalization"
CUB_ROOT = find_cub_root()
merge_previous_cub_outputs()

def find_checkpoint(model, fold):
    candidates = []
    rels = [
        Path("cub_outputs") / model / f"fold_{fold}" / "best.pt",
        Path(model) / f"fold_{fold}" / "best.pt",
    ]
    for r in input_roots():
        candidates += [r / rel for rel in rels]
    candidates += [WORK / rel for rel in rels]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find CUB checkpoint for {model} fold {fold}")

print("MODEL:", SELECTED_MODEL)
print("CODE:", CODE)
print("CUB_ROOT:", CUB_ROOT)
print("GPUs:", gpu_count())


In [ ]:

model = SELECTED_MODEL
jobs = []
for fold in FOLDS:
    out = WORK / "cub_outputs" / model / f"fold_{fold}"
    if (out / "best.pt").exists():
        print("Skip existing:", out / "best.pt")
        continue
    jobs.append((
        f"train {model} fold {fold}",
        [
            PY, CUB_CODE / "train_cub_model.py",
            "--model", model,
            "--cub-root", CUB_ROOT,
            "--fold", fold,
            "--output-dir", out,
            "--epochs", EPOCHS[model],
            "--batch-size", BATCH[model],
            "--lr", LR[model],
            "--weight-decay", 0.05,
            "--time-limit-minutes", 500,
        ],
    ))
    if model in BACKBONE_LR:
        jobs[-1][1].extend(["--backbone-lr", BACKBONE_LR[model]])
    if model in HEAD_LR:
        jobs[-1][1].extend(["--head-lr", HEAD_LR[model]])
    if model in WARMUP_EPOCHS:
        jobs[-1][1].extend(["--warmup-epochs", WARMUP_EPOCHS[model]])
    if model in PATIENCE:
        jobs[-1][1].extend(["--patience", PATIENCE[model]])
    if model in DROP_PATH:
        jobs[-1][1].extend(["--drop-path-rate", DROP_PATH[model]])

run_jobs(jobs, parallel_if_2gpu=PARALLEL_FOLDS_IF_2GPU)


In [ ]:

model = SELECTED_MODEL
for fold in FOLDS:
    out = WORK / "cub_eval" / model / f"fold_{fold}"
    if (out / "metrics_val.json").exists():
        print("Skip existing:", out / "metrics_val.json")
        continue
    run([
        PY, CUB_CODE / "evaluate_cub_model.py",
        "--model", model,
        "--checkpoint", find_checkpoint(model, fold),
        "--cub-root", CUB_ROOT,
        "--fold", fold,
        "--output-dir", out,
    ])

rows = []
for path in (WORK / "cub_eval" / model).glob("fold_*/metrics_val.json"):
    with open(path) as f:
        rows.append({"model": model, "fold": path.parent.name, **json.load(f)})
df = pd.DataFrame(rows)
if not df.empty:
    df["fold_idx"] = df["fold"].str.extract(r"(\d+)").astype(int)
    df = df.sort_values(["model", "fold_idx"]).drop(columns=["fold_idx"])
    df.to_csv(WORK / f"cub_classification_{model}_by_fold.csv", index=False)
    display(df.groupby("model")[["log_loss", "accuracy", "balanced_accuracy", "macro_f1", "top5_accuracy"]].agg(["mean", "std"]).round(4))


In [ ]:

model = SELECTED_MODEL
jobs = []
for fold in XAI_FOLDS:
    out = WORK / "cub_xai" / f"fold_{fold}" / model
    if (out / "xai_summary_cub.json").exists():
        print("Skip existing:", out / "xai_summary_cub.json")
        continue
    cmd = [
        PY, CUB_CODE / "run_cub_xai.py",
        "--model", model,
        "--checkpoint", find_checkpoint(model, fold),
        "--cub-root", CUB_ROOT,
        "--fold", fold,
        "--output-dir", out,
        "--max-samples", MAX_SAMPLES,
        "--methods", *METHODS,
    ]
    if SAVE_MAPS:
        cmd.append("--save-maps")
    if not RUN_FAITHFULNESS:
        cmd.append("--skip-faithfulness")
    if model in {"vit_small", "deit_small"}:
        cmd.append("--enable-attention-rollout")
    jobs.append((f"cub xai {model} fold {fold}", cmd))

run_jobs(jobs, parallel_if_2gpu=PARALLEL_FOLDS_IF_2GPU)

rows = []
for path in (WORK / "cub_xai").glob(f"fold_*/{model}/xai_summary_cub.json"):
    with open(path) as f:
        payload = json.load(f)
    s = payload["summary"]
    rows.append({
        "fold": path.parents[1].name,
        "model": model,
        "mean_spearman": s.get("mean_spearman"),
        "mean_top20_iou": s.get("mean_top20_iou"),
        "skipped": json.dumps(s.get("skipped", {}), sort_keys=True),
    })
xai_summary = pd.DataFrame(rows)
if not xai_summary.empty:
    xai_summary["fold_idx"] = xai_summary["fold"].str.extract(r"(\d+)").astype(int)
    xai_summary = xai_summary.sort_values(["model", "fold_idx"]).drop(columns=["fold_idx"])
    xai_summary.to_csv(WORK / f"cub_xai_summary_{model}_by_fold.csv", index=False)
    display(xai_summary)
